In [23]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist

import matplotlib.pyplot as plt

In [24]:
PROJECT_ROOT = Path.cwd().parent

DATA_ROOT = PROJECT_ROOT / "data/sample"

SAMPLE_ID = "44b6_0113de3b"

PROCESSED_DIR = (
        DATA_ROOT
        / "processed"
        / "stage_7_processed_dataset"
        / SAMPLE_ID
)

FEATURE_DIR = PROCESSED_DIR / "cells"

In [25]:
cell_files = sorted(FEATURE_DIR.glob("t*.csv"))

time_frames = [
    pd.read_csv(file)
    for file in cell_files
]

print(f"Loaded {len(time_frames)} timepoints.")

Loaded 20 timepoints.


In [26]:
time_frames[0].head()

,cell_id,volume_voxels,z_min,y_min,x_min,z_max,y_max,x_max,centroid_z,centroid_y,...,equivalent_radius,axis_major,axis_middle,axis_minor,elongation,flatness,anisotropy,convex_volume,solidity,compactness
0,1,186.0,0,0,46,2,14,60,0.231183,6.354839,...,3.541127,3.312989,2.928501,0.420773,1.131292,6.959811,7.873577,77.166667,2.410367,0.023965
1,2,222.0,0,0,65,2,16,80,0.153153,6.815315,...,3.756253,4.082647,3.323817,0.354509,1.228301,9.375832,11.516344,88.000000,2.522727,0.021110
2,3,240.0,0,18,59,4,35,72,0.320833,26.012500,...,3.855146,3.955908,2.965969,0.548120,1.333766,5.411165,7.217226,171.666667,1.398058,0.009131
3,4,737.0,0,55,48,4,72,68,1.074627,62.743555,...,5.603503,4.670576,3.679390,0.964733,1.269389,3.813895,4.841316,520.500000,1.415946,0.014695
4,5,343.0,0,89,28,2,109,43,0.364431,98.571429,...,4.342453,4.462224,3.304942,0.480760,1.350167,6.874411,9.281601,154.833333,2.215285,0.023338


In [27]:
for t, df in enumerate(time_frames):
    print(f"t={t:03d}: {len(df)} cells")

t=000: 202 cells
t=001: 211 cells
t=002: 209 cells
t=003: 214 cells
t=004: 211 cells
t=005: 199 cells
t=006: 207 cells
t=007: 209 cells
t=008: 205 cells
t=009: 206 cells
t=010: 209 cells
t=011: 208 cells
t=012: 215 cells
t=013: 220 cells
t=014: 222 cells
t=015: 222 cells
t=016: 206 cells
t=017: 211 cells
t=018: 213 cells
t=019: 224 cells


In [28]:
MAX_DISTANCE = 12.0

# -----------------------------
# Cost function weights
# -----------------------------
W_DISTANCE = 0.8
W_VOLUME = 0.2
W_INTENSITY = 0.10

W_INTENSITY_MEAN = 0.70
W_INTENSITY_STD = 0.30

W_DISTANCE = 0.40
W_SIZE = 0.20
W_SHAPE = 0.25
W_INTENSITY = 0.10
W_BBOX = 0.05

# Reject matches whose volumes differ by more than this ratio
MAX_VOLUME_RATIO = 1.5

# Number of times to re-estimate the global shift using the current
# set of accepted matches before finalizing assignment for a frame pair.
SHIFT_REFINE_ITERS = 3

In [29]:
def robust_shift(coords0, coords1, rows=None, cols=None):
    """
    Estimate the global shift between two point sets.

    If `rows`/`cols` (matched pairs from a previous assignment pass) are
    supplied, the shift is the median displacement over just those
    matched pairs. This is robust to any cells that are new, lost, or
    otherwise didn't match last pass, since they're simply excluded.

    Otherwise, falls back to a coarse median-of-all-points estimate,
    which is already far less sensitive than a mean to a few new/lost
    cells (mitosis, cells entering/leaving the field of view, spurious
    detections, etc.).
    """
    if rows is not None and len(rows) > 0:
        disp = coords1[cols] - coords0[rows]
        return np.median(disp, axis=0)

    return np.median(coords1, axis=0) - np.median(coords0, axis=0)

In [30]:
from scipy.spatial.distance import cdist
from scipy.optimize import linear_sum_assignment
import numpy as np


def assign(
        coords0_shifted,
        coords1,
        detections0,
        detections1,
):
    """
    Build the assignment cost matrix using feature groups.

    Feature Groups
    --------------
    • Distance
    • Size
    • Shape
    • Intensity
    • Bounding Box
    """

    EPS = 1e-8

    # ==========================================================
    # Feature Groups
    # ==========================================================

    SIZE_FEATURES = [
        "volume",
        "extent",
        "equivalent_radius",
    ]

    SHAPE_FEATURES = [
        "elongation",
        "flatness",
        "anisotropy",
        "solidity",
        "compactness",
    ]

    INTENSITY_FEATURES = [
        "intensity_mean",
        "intensity_std",
        "intensity_cv",
    ]

    BBOX_FEATURES = [
        "bbox_depth",
        "bbox_height",
        "bbox_width",
    ]

    # ==========================================================
    # Helper
    # ==========================================================

    def feature_group_cost(feature_names):
        """
        Average normalized difference over a feature group.
        """

        total = None

        for feature in feature_names:

            a = detections0[feature].to_numpy(dtype=float)
            b = detections1[feature].to_numpy(dtype=float)

            cost = (
                    np.abs(a[:, None] - b[None, :])
                    / (np.maximum(a[:, None], b[None, :]) + EPS)
            )

            cost = np.nan_to_num(
                cost,
                nan=1.0,
                posinf=1.0,
                neginf=1.0,
            )

            if total is None:
                total = cost
            else:
                total += cost

        return total / len(feature_names)

    # ==========================================================
    # Distance
    # ==========================================================

    distance_matrix = cdist(coords0_shifted, coords1)

    distance_cost = distance_matrix / MAX_DISTANCE

    # ==========================================================
    # Feature Group Costs
    # ==========================================================

    size_cost = feature_group_cost(SIZE_FEATURES)

    shape_cost = feature_group_cost(SHAPE_FEATURES)

    intensity_cost = feature_group_cost(INTENSITY_FEATURES)

    bbox_cost = feature_group_cost(BBOX_FEATURES)

    # ==========================================================
    # Hard Constraints
    # ==========================================================

    vol0 = detections0["volume"].to_numpy(dtype=float)
    vol1 = detections1["volume"].to_numpy(dtype=float)

    volume_ratio = (
            np.maximum(vol0[:, None], vol1[None, :])
            / (np.minimum(vol0[:, None], vol1[None, :]) + EPS)
    )

    invalid = (
            (distance_matrix > MAX_DISTANCE)
            | (volume_ratio > MAX_VOLUME_RATIO)
    )

    # ==========================================================
    # Final Cost
    # ==========================================================

    cost_matrix = (

            W_DISTANCE * distance_cost

            + W_SIZE * size_cost

            + W_SHAPE * shape_cost

            + W_INTENSITY * intensity_cost

            + W_BBOX * bbox_cost

    )

    cost_matrix[invalid] = 1e6

    rows, cols = linear_sum_assignment(cost_matrix)

    valid = ~invalid[rows, cols]

    return (
        rows[valid],
        cols[valid],
        distance_matrix,
    )

In [31]:
# ------------------------------------------------------------
# Initialize tracks using the first frame
# ------------------------------------------------------------

current_track_ids = np.arange(len(time_frames[0]), dtype=int)
next_track_id = len(current_track_ids)

tracks = []

for cell, track in enumerate(current_track_ids):

    row = time_frames[0].iloc[cell]

    tracks.append({
        "track_id": track,
        "frame": 0,
        "cell": cell,
        "z": row["centroid_z"],
        "y": row["centroid_y"],
        "x": row["centroid_x"],
        "volume": row["volume_voxels"],
    })

# ------------------------------------------------------------
# Process every pair of frames
# ------------------------------------------------------------

for t in range(len(time_frames) - 1):

    detection0 = time_frames[t]
    detection1 = time_frames[t + 1]

    # -----------------------------
    # Coordinates / volumes
    # -----------------------------

    coords0 = detection0[
        ["centroid_z", "centroid_y", "centroid_x"]
    ].to_numpy(dtype=float)

    coords1 = detection1[
        ["centroid_z", "centroid_y", "centroid_x"]
    ].to_numpy(dtype=float)

    vol0 = detection0["volume_voxels"].to_numpy(dtype=float)
    vol1 = detection1["volume_voxels"].to_numpy(dtype=float)

    # -----------------------------
    # Estimate global motion, then refine it using the cells that
    # actually matched last pass. This stops a handful of divergent
    # cells (new/lost cells, noise, mitosis) from biasing the shift
    # estimate used for every other cell in the frame.
    # -----------------------------

    global_shift = robust_shift(coords0, coords1)
    rows, cols, distance_matrix = np.array([], dtype=int), np.array([], dtype=int), None

    for _ in range(SHIFT_REFINE_ITERS):

        predicted_coords0 = coords0 + global_shift

        rows, cols, distance_matrix = assign(
            predicted_coords0,
            coords1,
            detection0,
            detection1,
        )

        if len(rows) == 0:
            break

        refined_shift = robust_shift(coords0, coords1, rows, cols)

        converged = np.allclose(refined_shift, global_shift, atol=1e-3)
        global_shift = refined_shift

        if converged:
            break

    print(
        f"Frame {t}->{t+1} "
        f"shift = ({global_shift[0]:.2f}, "
        f"{global_shift[1]:.2f}, "
        f"{global_shift[2]:.2f})"
    )

    # -----------------------------
    # Matches are already filtered for distance/volume validity
    # inside assign(), so no further thresholding is needed here.
    # -----------------------------

    accepted = pd.DataFrame({
        "cell_t": rows,
        "cell_t1": cols,
        "distance": distance_matrix[rows, cols] if len(rows) else [],
    })

    next_track_ids = np.full(
        len(detection1),
        -1,
        dtype=int,
    )

    # --------------------------------------------------------
    # Continue existing tracks
    # --------------------------------------------------------

    for _, row in accepted.iterrows():

        prev_cell = int(row.cell_t)
        next_cell = int(row.cell_t1)

        next_track_ids[next_cell] = current_track_ids[prev_cell]

    # --------------------------------------------------------
    # Start new tracks
    # --------------------------------------------------------

    for cell in np.where(next_track_ids == -1)[0]:

        next_track_ids[cell] = next_track_id
        next_track_id += 1

    # --------------------------------------------------------
    # Save tracks
    # --------------------------------------------------------

    for cell, track in enumerate(next_track_ids):

        row = detection1.iloc[cell]

        tracks.append({
            "track_id": track,
            "frame": t + 1,
            "cell": cell,
            "z": row["centroid_z"],
            "y": row["centroid_y"],
            "x": row["centroid_x"],
            "volume": row["volume_voxels"],
        })

    print(
        f"{t:03d}->{t+1:03d} : "
        f"{len(accepted):3d} matches | "
        f"{len(detection1) - len(accepted):2d} new tracks"
    )

    current_track_ids = next_track_ids

tracks = pd.DataFrame(tracks)

Frame 0->1 shift = (0.68, 2.99, 1.91)
000->001 : 169 matches | 42 new tracks
Frame 1->2 shift = (0.85, 0.91, 0.78)
001->002 : 172 matches | 37 new tracks
Frame 2->3 shift = (0.84, 1.79, 3.87)
002->003 : 165 matches | 49 new tracks
Frame 3->4 shift = (1.19, 2.39, 4.03)
003->004 : 172 matches | 39 new tracks
Frame 4->5 shift = (0.83, 1.99, 1.05)
004->005 : 168 matches | 31 new tracks
Frame 5->6 shift = (0.75, 4.62, -1.06)
005->006 : 126 matches | 81 new tracks
Frame 6->7 shift = (0.98, 2.39, 2.05)
006->007 : 171 matches | 38 new tracks
Frame 7->8 shift = (0.75, 2.84, 3.31)
007->008 : 178 matches | 27 new tracks
Frame 8->9 shift = (0.72, 3.26, 1.14)
008->009 : 174 matches | 32 new tracks
Frame 9->10 shift = (1.20, -2.11, 0.96)
009->010 : 179 matches | 30 new tracks
Frame 10->11 shift = (0.49, 4.44, 1.71)
010->011 : 158 matches | 50 new tracks
Frame 11->12 shift = (0.26, 8.09, -0.15)
011->012 : 179 matches | 36 new tracks
Frame 12->13 shift = (-0.34, 5.64, -3.64)
012->013 : 175 matches | 4

In [32]:
import pandas as pd

tracks = pd.DataFrame(tracks)

Save results

In [33]:
import json

# ------------------------------------------------------------
# Save cell tracks
# ------------------------------------------------------------

output_dir = (
        DATA_ROOT
        / "processed"
        / "stage_8_cell_tracking"
)

output_dir.mkdir(parents=True, exist_ok=True)

tracks.to_csv(
    output_dir / "tracks.csv",
    index=False,
)

print(f"Saved {len(tracks):,} track records.")
print(f"Output: {output_dir / 'tracks.csv'}")

# ------------------------------------------------------------
# Save tracking parameters
# ------------------------------------------------------------

metadata = {
    "max_distance": MAX_DISTANCE,
    "distance_weight": W_DISTANCE,
    "volume_weight": W_VOLUME,
    "max_volume_ratio": MAX_VOLUME_RATIO,
}

with open(output_dir / "metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)

print(f"Saved Stage 8 results to: {output_dir}")

Saved 4,223 track records.
Output: D:\Projects\Kaggle\cell-tracking\data\sample\processed\stage_8_cell_tracking\tracks.csv
Saved Stage 8 results to: D:\Projects\Kaggle\cell-tracking\data\sample\processed\stage_8_cell_tracking
